# Mancala (Kalah) Implementation

## Includes: plain alpha-beta minimax with store evaluation, heuristic minimax using H1, H4, H6, H7 (based on Hunter's 2021 paper from our research).

In [1]:
class MancalaBoard:
  def __init__(self):
      self.board = [
          [4, 4, 4, 4, 4, 4, 0],
          [4, 4, 4, 4, 4, 4, 0],
      ]
      self.current_player = 0

  def copy(self):
      new_board = MancalaBoard()
      new_board.board = [row[:] for row in self.board]
      new_board.current_player = self.current_player
      return new_board

  def get_legal_moves(self, player):
    # Returns pits that aren't empty (indices 0 to 5)
    return [i for i in range(6) if self.board[player][i] > 0]

  def is_terminal(self):
    # Game ends when either player's pits are all empty
    return (all(self.board[0][i] == 0 for i in range(6)) or
          all(self.board[1][i] == 0 for i in range(6)))

  def collect_remaining(self):
      for player in range(2):
          for pit in range(6):
              self.board[player][6] += self.board[player][pit]
              self.board[player][pit] = 0

  def distribute_stones(self, player, pit):
      stones = self.board[player][pit]
      self.board[player][pit] = 0
      current_side = player
      current_pit = pit

      while stones > 0:
          current_pit += 1
          if current_pit == 7:
              current_side = 1 - current_side
              current_pit = 0
          # skip opponent's store
          if current_pit == 6 and current_side != player:
              current_side = 1 - current_side
              current_pit = 0
          # drop stone in pit
          self.board[current_side][current_pit] += 1
          stones -= 1

      # bonus turn! yes!!!
      if current_side == player and current_pit == 6:
          return True

      # "empty capture rule", where last stone lands in empty pit
      if (current_side == player and current_pit < 6
              and self.board[player][current_pit] == 1):
          opposite_pit = 5 - current_pit
          captured = self.board[1 - player][opposite_pit]
          if captured > 0:
              self.board[player][6] += captured + 1
              self.board[player][current_pit] = 0
              self.board[1 - player][opposite_pit] = 0
      return False

  def make_move(self, pit):
      player = self.current_player
      bonus_turn = self.distribute_stones(player, pit)
      if not bonus_turn:
          self.current_player = 1 - player
      if self.is_terminal():
          self.collect_remaining()
      return self

  def __str__(self):
      p0, p1 = self.board[0], self.board[1]
      top = "   " + "   ".join(f"{v:>2}" for v in reversed(p1[:6])) + "   "
      bot = "   " + "   ".join(f"{v:>2}" for v in p0[:6])     + "   "
      mid = f"{p1[6]:>2}" + " " * (len(top) - 4) + f"{p0[6]:>2}"
      return f"{top}\n{mid}\n{bot}"

In [2]:
# check start board, legal moves, is terminal
game = MancalaBoard()
print(game)
print("Legal moves for P0:", game.get_legal_moves(0))
print("Is terminal:", game.is_terminal())

# total stones print (must always be 48)
g = MancalaBoard()
g.make_move(2)
total = sum(g.board[0]) + sum(g.board[1])
print(f"Total stones after one move: {total} (must be 48)")

    4    4    4    4    4    4   
 0                              0
    4    4    4    4    4    4   
Legal moves for P0: [0, 1, 2, 3, 4, 5]
Is terminal: False
Total stones after one move: 48 (must be 48)


In [3]:
# plain alpha-beta minimax with store point difference evaluation
def evaluate(board, player):
    # store 'differential' (H4 alone, used as the plain alpha-beta baseline)
    return board.board[player][6] - board.board[1 - player][6]

def alpha_beta(board, depth, alpha, beta, player, evaluate_fn=None):
    if evaluate_fn is None:
        evaluate_fn = evaluate

    if board.is_terminal() or depth == 0:
        if board.is_terminal():
            board.collect_remaining()
        return evaluate_fn(board, player)

    current = board.current_player

    if current == player:  # maximizing
        best_value = float('-inf')
        for move in board.get_legal_moves(current):
            new_board = board.copy()
            new_board.make_move(move)
            # bonus-turn!!! rule
            new_depth = depth if new_board.current_player == current else depth - 1
            value = alpha_beta(new_board, new_depth, alpha, beta, player, evaluate_fn)
            best_value = max(best_value, value)
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break  # prune
        return best_value

    else:  # minimizing
        best_value = float('inf')
        for move in board.get_legal_moves(current):
            new_board = board.copy()
            new_board.make_move(move)
            new_depth = depth if new_board.current_player == current else depth - 1
            value = alpha_beta(new_board, new_depth, alpha, beta, player, evaluate_fn)
            best_value = min(best_value, value)
            beta = min(beta, best_value)
            if beta <= alpha:
                break
        return best_value


def minimax_agent(board, depth=8, evaluate_fn=None):
    if evaluate_fn is None:
        evaluate_fn = evaluate
    player = board.current_player
    best_move = None
    best_value = float('-inf')

    for move in board.get_legal_moves(player):
        new_board = board.copy()
        new_board.make_move(move)
        new_depth = depth if new_board.current_player == player else depth - 1
        value = alpha_beta(new_board, new_depth, float('-inf'), float('inf'),
                           player, evaluate_fn)
        if value > best_value:
            best_value = value
            best_move = move

    return best_move

In [4]:
# minimax check
game = MancalaBoard()
move = minimax_agent(game, depth=4)
print(f"Best move for P0 with plain alpha-beta at depth 4: pit {move}")

Best move for P0 with plain alpha-beta at depth 4: pit 5


In [5]:
#heuristics (we selected H1, H4, H6, H7 for brevity)
def H1(board, player):
    """Hoard seeds in the leftmost pit (furthest from own store)."""
    return board.board[player][0]

def H4(board, player):
    """Number of seeds in own store."""
    return board.board[player][6]

def H6(board, player):
    """Negative of opponent's store; we want this small."""
    return -board.board[1 - player][6]

def H7(board, player):
    """1 if it is still player's turn at this state (last move earned a bonus turn)."""
    return 1 if board.current_player == player else 0

# Weights from Hunter (2021) Table 3 averages
W1, W4, W6, W7 = 0.2, 1.0, 0.6, 0.9

def evaluate_heuristic(board, player):
    return (W1 * H1(board, player)
          + W4 * H4(board, player)
          + W6 * H6(board, player)
          + W7 * H7(board, player))

In [6]:
#heuristics check
import time
b = MancalaBoard()
print(f"Start position H1={H1(b,0)}, H4={H4(b,0)}, H6={H6(b,0)}, H7={H7(b,0)}, "
      f"weighted={evaluate_heuristic(b,0):.2f}")

t0 = time.time()
m1 = minimax_agent(MancalaBoard(), depth=6)
print(f"Plain alpha-beta picks pit {m1} in {time.time()-t0:.2f}s")

t0 = time.time()
m2 = minimax_agent(MancalaBoard(), depth=6, evaluate_fn=evaluate_heuristic)
print(f"Heuristic minimax picks pit {m2} in {time.time()-t0:.2f}s")

Start position H1=4, H4=0, H6=0, H7=1, weighted=1.70
Plain alpha-beta picks pit 2 in 0.67s
Heuristic minimax picks pit 5 in 0.92s


In [7]:
# heuristic minimax (P0) vs plain alpha-beta (P1)
g = MancalaBoard()
print("Starting position:")
print(g)

move_num = 0
while not g.is_terminal():
    fn = evaluate_heuristic if g.current_player == 0 else evaluate
    move = minimax_agent(g, depth=4, evaluate_fn=fn)
    if move is None:
        break
    move_num += 1
    print(f"\n--- Move {move_num}: P{g.current_player} plays pit {move} ---")
    g.make_move(move)
    print(g)

g.collect_remaining()
total = g.board[0][6] + g.board[1][6]
print(f"\nFinal: P0 (heuristic) = {g.board[0][6]}, P1 (plain) = {g.board[1][6]}, "
      f"total = {total}")

Starting position:
    4    4    4    4    4    4   
 0                              0
    4    4    4    4    4    4   

--- Move 1: P0 plays pit 5 ---
    4    4    4    5    5    5   
 0                              1
    4    4    4    4    4    0   

--- Move 2: P1 plays pit 1 ---
    5    5    5    6    0    5   
 1                              1
    4    4    4    4    4    0   

--- Move 3: P1 plays pit 0 ---
    6    6    6    7    1    0   
 1                              1
    4    4    4    4    4    0   

--- Move 4: P0 plays pit 2 ---
    6    6    6    7    1    0   
 1                              2
    4    4    0    5    5    1   

--- Move 5: P0 plays pit 5 ---
    6    6    6    7    1    0   
 1                              3
    4    4    0    5    5    0   

--- Move 6: P0 plays pit 4 ---
    6    6    6    8    2    1   
 1                              4
    4    4    0    5    0    1   

--- Move 7: P1 plays pit 1 ---
    6    6    7    9    0    1   
 1       

In [ ]:
# heuristic minimax (P0) vs random agent (P1) (baseline)
import random
def random_agent(board):
    """Pick a uniformly random legal move from the current player's pits."""
    legal_moves = board.get_legal_moves(board.current_player)
    return random.choice(legal_moves) if legal_moves else None

g = MancalaBoard()
while not g.is_terminal():
    if g.current_player == 0:
        move = minimax_agent(g, depth=4, evaluate_fn=evaluate_heuristic)
    else:
        move = random_agent(g)
    if move is None:
        break
    g.make_move(move)
    print(g)
g.collect_remaining()
print(f"Heuristic (P0) {g.board[0][6]}  vs  Random (P1) {g.board[1][6]}")

    4    4    4    5    5    5   
 0                              1
    4    4    4    4    4    0   
    5    0    4    5    5    5   
 1                              1
    5    5    4    4    4    0   
    5    0    4    5    5    5   
 1                              2
    5    5    0    5    5    1   
    5    0    4    5    5    5   
 1                              3
    5    5    0    5    5    0   
    5    0    4    5    5    5   
 1                              4
    5    0    1    6    6    1   
    5    0    4    5    5    5   
 1                              5
    5    0    1    6    6    0   
    5    0    4    5    5    0   
 1                             11
    0    1    2    7    7    0   
    6    1    5    0    5    0   
 2                             11
    1    1    2    7    7    0   
    6    1    6    1    6    1   
 2                             12
    1    1    2    0    8    1   
    7    2    0    1    6    1   
 3                             12
    2    2    